<a href="https://colab.research.google.com/github/contemplator/nchu-research-method-traffic-violation-dataset/blob/feat/458-images/notebooks/NCHU_Research_YOLOv7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **國立中興大學 研究方法 (Research Methodology) - 期末專案**
### **基於 YOLOv7 消融實驗之台灣道路交通設施與違規偵測研究 (458 張完整資料集 Exp 1 ~ Exp 10)**
本 Notebook 實作 `docs/NEXT_STAGE_EXPERIMENT_PLAN.md` 規劃之 **Append-Only 消融實驗架構**：
- **資料集**：458 張影像（Train 320 / Val 91 / Test 47，6 類別 + 無目標負樣本）。
- **基準保留 (`Exp 1 ~ Exp 6`)**：完整保留輕量 Tiny 與大骨幹 Base 之歷史基準與超參數演化脈絡。
- **進階消融 (`Exp 7 ~ Exp 10`)**：Base-100ep 預設對照 ➔ 960 高解析度 ➔ 領域調優 v2 (Focal Loss + 關閉水平翻轉 + Box 加權) ➔ LAB-CLAHE 影像前處理。
- **嚴格防資料洩漏**：先於 Val 集尋找最佳 F1 信心門檻 `best_conf`，再統一於 Test 集評估與產出 4 維度誤差診斷與並排推論對比矩陣。
- **自動跳過已訓練權重**：若 Google Drive 的 `checkpoints/exp1~exp10` 已存在 `best.pt`，Step 5 會自動跳過訓練並還原權重與日誌。
- **動態影片推論對比 (`Step 10`)**：可直接讀取 Google Drive 的 `exp7`（640px）與 `exp8`（960px）權重製作左右並排對比影片（`comparison_demo.mp4` / `.gif`），另存至 Google Drive 獨立目錄 `demo_videos/` 供 PPT 嵌入展示（執行完 Step 1~3 後即可直接跳至 Step 10）。

# **Step 1: 環境檢查與 GPU 配置 (Environment Checking)**

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 是否可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 型號: {torch.cuda.get_device_name(0)}")

# **Step 2: 掛載 Google 雲端硬碟 (Mount Google Drive)**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# **Step 3: 下載專案倉庫與安裝 YOLOv7 依賴 (Setup YOLOv7)**
我們將從 GitHub 克隆您的模型訓練庫與 `feat/458-images` 資料集分支。

In [ ]:
%cd /content
!mkdir -p workspace
%cd /content/workspace

![ ! -d "yolov7" ] && git clone https://github.com/contemplator/nchu-research-method-traffic-violation-model.git yolov7
![ ! -d "dataset_repo" ] && git clone -b feat/458-images https://github.com/contemplator/nchu-research-method-traffic-violation-dataset.git dataset_repo
!ln -sfn /content/workspace/dataset_repo /content/workspace/nchu-research-method-traffic-violation-dataset

%cd /content/workspace/yolov7
!pip install -q thop albumentations

import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

# 1. 自動修復 NumPy 2.x 相容性問題 (NumPy 2.0+ 移除 np.trapz 改為 np.trapezoid)
metrics_file = "/content/workspace/yolov7/utils/metrics.py"
if os.path.exists(metrics_file):
    with open(metrics_file, "r", encoding="utf-8") as f:
        content = f.read()
    if "np.trapz" in content:
        content = content.replace("np.trapz", 'getattr(np, "trapezoid", getattr(np, "trapz", None))')
        with open(metrics_file, "w", encoding="utf-8") as f:
            f.write(content)
        print("✅ 已成功修復 yolov7/utils/metrics.py 的 NumPy 2.x 相容性！")

# 2. 自動修復 google_utils.py 在 GitHub API 限流時引發 KeyError: "assets"
google_utils_file = "/content/workspace/yolov7/utils/google_utils.py"
if os.path.exists(google_utils_file):
    with open(google_utils_file, "r", encoding="utf-8") as f:
        g_content = f.read()
    if "response['assets']" in g_content:
        g_content = g_content.replace("response['assets']", "response.get('assets', [])")
        with open(google_utils_file, "w", encoding="utf-8") as f:
            f.write(g_content)
        print("✅ 已成功修復 yolov7/utils/google_utils.py 的 GitHub API 防呆機制！")

## **下載官方預訓練權重 (Pretrained Weights)**

In [ ]:
%cd /content/workspace/yolov7
!wget -nc -P /content/workspace/yolov7 https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7-tiny.pt
!wget -nc -P /content/workspace/yolov7 https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7.pt

# **Step 4: 資料集解壓、四道品質防線驗證與 CLAHE 前處理**
1. 優先從 Google Drive（或本機 repo 內建的 `data/dataset.zip`）解壓 458 張切分資料集（320 / 91 / 47）。
2. 執行 `scripts/verify_annotations.py --dataset_dir data/dataset` 檢查檔案完整度、類別分佈與負樣本比例。
3. 執行 `scripts/preprocess_clahe.py` 產出獨立的 `data/dataset_clahe/` 與 `configs/traffic_facility_clahe.yaml` 供 Exp 10 使用。

In [ ]:
import os

DRIVE_DATASET_ZIP = "/content/drive/MyDrive/NCHU_YOLO_Project_Traffic_Facility_6_classes/dataset.zip"
REPO_DATASET_ZIP = "/content/workspace/dataset_repo/data/dataset.zip"
TARGET_DATA_DIR = "/content/workspace/dataset_repo/data"

zip_to_use = DRIVE_DATASET_ZIP if os.path.exists(DRIVE_DATASET_ZIP) else REPO_DATASET_ZIP
if os.path.exists(zip_to_use):
    !unzip -o -q {zip_to_use} -d {TARGET_DATA_DIR}
    print(f"✅ 資料集解壓縮完成：{zip_to_use}")
else:
    print("⚠️ 找不到 dataset.zip，請確認 Google Drive 或 dataset_repo/data/dataset.zip 是否存在。")

%cd /content/workspace/dataset_repo
!python scripts/verify_annotations.py --dataset_dir data/dataset
!python scripts/preprocess_clahe.py --src data/dataset --dst data/dataset_clahe --yaml-out configs/traffic_facility_clahe.yaml
!cat configs/traffic_facility.yaml

# **Step 5: 結構化消融實驗登錄表 (`Exp 1 ~ Exp 10`) 與自動化訓練迴圈**
- 使用 `EXPERIMENTS` 清單管理所有實驗，透過 `"enabled": True / False` 控制是否執行（預設可依算力分批或全開）。
- **支援自動跳過已訓練實驗 (`SKIP_IF_CKPT_EXISTS = True`)**：若 Google Drive `checkpoints/{exp_id}/best.pt` 已存在，會自動跳過 `train.py` 並將權重與日誌還原至 `runs/train/{exp_name}/`，讓後續評估流程無縫銜接。
- 每組實驗訓練完成後，立即將 `best.pt`、`results.txt`、`results.png` 與 `confusion_matrix.png` 備份至 Google Drive，防止 Colab 斷線遺失進度。

In [ ]:
import os
import shutil

os.environ["WANDB_MODE"] = "disabled"
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
%cd /content/workspace/yolov7

DRIVE_CKPT_ROOT = "/content/drive/MyDrive/NCHU_YOLO_Project_Traffic_Facility_6_classes/checkpoints"
os.makedirs(DRIVE_CKPT_ROOT, exist_ok=True)
SKIP_IF_CKPT_EXISTS = True  # 若 Google Drive 已有該實驗的 best.pt，則跳過訓練並直接還原權重與日誌

DATA_STD = "/content/workspace/dataset_repo/configs/traffic_facility.yaml"
DATA_CLAHE = "/content/workspace/dataset_repo/configs/traffic_facility_clahe.yaml"
CFG_TINY = "/content/workspace/dataset_repo/configs/yolov7-tiny-custom.yaml"
CFG_BASE = "cfg/training/yolov7.yaml"

EXPERIMENTS = [
    {
        "id": "exp1",
        "name": "exp1_yolov7_tiny_baseline",
        "label": "Exp 1: Tiny-640 (50ep)",
        "short_label": "Exp1 Tiny-50ep",
        "arch": "YOLOv7-tiny",
        "img_size": 640,
        "batch": 16,
        "epochs": 50,
        "cfg": CFG_TINY,
        "weights": "yolov7-tiny.pt",
        "data": DATA_STD,
        "hyp": "data/hyp.scratch.tiny.yaml",
        "is_clahe": False,
        "enabled": True,
    },
    {
        "id": "exp2",
        "name": "exp2_yolov7_base_scale",
        "label": "Exp 2: Base-640 (50ep)",
        "short_label": "Exp2 Base-50ep",
        "arch": "YOLOv7 Base",
        "img_size": 640,
        "batch": 8,
        "epochs": 50,
        "cfg": CFG_BASE,
        "weights": "yolov7.pt",
        "data": DATA_STD,
        "hyp": "data/hyp.scratch.p5.yaml",
        "is_clahe": False,
        "enabled": True,
    },
    {
        "id": "exp3",
        "name": "exp3_yolov7_tiny_100ep",
        "label": "Exp 3: Tiny-640 (100ep)",
        "short_label": "Exp3 Tiny-100ep",
        "arch": "YOLOv7-tiny",
        "img_size": 640,
        "batch": 16,
        "epochs": 100,
        "cfg": CFG_TINY,
        "weights": "yolov7-tiny.pt",
        "data": DATA_STD,
        "hyp": "data/hyp.scratch.tiny.yaml",
        "is_clahe": False,
        "enabled": True,
    },
    {
        "id": "exp4",
        "name": "exp4_yolov7_tiny_original_hyp",
        "label": "Exp 4: Tiny-640 (orig hyp)",
        "short_label": "Exp4 Tiny-orig",
        "arch": "YOLOv7-tiny",
        "img_size": 640,
        "batch": 16,
        "epochs": 100,
        "cfg": CFG_TINY,
        "weights": "yolov7-tiny.pt",
        "data": DATA_STD,
        "hyp": "/content/workspace/dataset_repo/configs/hyp.scratch.custom.yaml.original",
        "is_clahe": False,
        "enabled": True,
    },
    {
        "id": "exp5",
        "name": "exp5_yolov7_tiny_tuned_v1",
        "label": "Exp 5: Tiny-640 (tuned v1)",
        "short_label": "Exp5 Tiny-v1",
        "arch": "YOLOv7-tiny",
        "img_size": 640,
        "batch": 16,
        "epochs": 100,
        "cfg": CFG_TINY,
        "weights": "yolov7-tiny.pt",
        "data": DATA_STD,
        "hyp": "/content/workspace/dataset_repo/configs/hyp.scratch.custom.yaml",
        "is_clahe": False,
        "enabled": True,
    },
    {
        "id": "exp6",
        "name": "exp4_yolov7_tiny_tuned",  # 保留原路徑名稱以相容舊腳本
        "label": "Exp 6: Base-640 (tuned v1)",
        "short_label": "Exp6 Base-v1",
        "arch": "YOLOv7 Base",
        "img_size": 640,
        "batch": 8,
        "epochs": 100,
        "cfg": CFG_BASE,
        "weights": "yolov7.pt",
        "data": DATA_STD,
        "hyp": "/content/workspace/dataset_repo/configs/hyp.scratch.custom.yaml",
        "is_clahe": False,
        "enabled": True,
    },
    {
        "id": "exp7",
        "name": "exp7_yolov7_base_100ep_default",
        "label": "Exp 7: Base-640 (100ep default)",
        "short_label": "Exp7 Base-def",
        "arch": "YOLOv7 Base",
        "img_size": 640,
        "batch": 8,
        "epochs": 100,
        "cfg": CFG_BASE,
        "weights": "yolov7.pt",
        "data": DATA_STD,
        "hyp": "data/hyp.scratch.p5.yaml",
        "is_clahe": False,
        "enabled": True,
    },
    {
        "id": "exp8",
        "name": "exp8_yolov7_base_960_highres",
        "label": "Exp 8: Base-960 (highres v1)",
        "short_label": "Exp8 Base-960",
        "arch": "YOLOv7 Base",
        "img_size": 960,
        "batch": 8,
        "epochs": 100,
        "cfg": CFG_BASE,
        "weights": "yolov7.pt",
        "data": DATA_STD,
        "hyp": "/content/workspace/dataset_repo/configs/hyp.traffic_highres.yaml",
        "is_clahe": False,
        "enabled": True,
    },
    {
        "id": "exp9",
        "name": "exp9_yolov7_base_960_v2",
        "label": "Exp 9: Base-960 (tuned v2)",
        "short_label": "Exp9 Base-v2",
        "arch": "YOLOv7 Base",
        "img_size": 960,
        "batch": 8,
        "epochs": 100,
        "cfg": CFG_BASE,
        "weights": "yolov7.pt",
        "data": DATA_STD,
        "hyp": "/content/workspace/dataset_repo/configs/hyp.traffic_v2.yaml",
        "is_clahe": False,
        "enabled": True,
    },
    {
        "id": "exp10",
        "name": "exp10_yolov7_base_960_v2_clahe",
        "label": "Exp 10: Base-960 (v2 + CLAHE)",
        "short_label": "Exp10 CLAHE",
        "arch": "YOLOv7 Base",
        "img_size": 960,
        "batch": 8,
        "epochs": 100,
        "cfg": CFG_BASE,
        "weights": "yolov7.pt",
        "data": DATA_CLAHE,
        "hyp": "/content/workspace/dataset_repo/configs/hyp.traffic_v2.yaml",
        "is_clahe": True,
        "enabled": True,
    },
]

for exp in EXPERIMENTS:
    if not exp["enabled"]:
        print(f"⏭️ 略過未啟用實驗: {exp['label']}")
        continue

    run_dir = f"runs/train/{exp['name']}"
    ckpt_dst = os.path.join(DRIVE_CKPT_ROOT, exp["id"])
    drive_best_pt = os.path.join(ckpt_dst, "best.pt")

    if SKIP_IF_CKPT_EXISTS and os.path.exists(drive_best_pt):
        print(f"⏭️ 偵測到 Google Drive 已有 {exp['id']} 權重，跳過訓練並還原至 {run_dir}")
        os.makedirs(os.path.join(run_dir, "weights"), exist_ok=True)
        shutil.copy2(drive_best_pt, os.path.join(run_dir, "weights", "best.pt"))
        for fname in ["results.txt", "results.png", "confusion_matrix.png"]:
            src_ckpt = os.path.join(ckpt_dst, fname)
            if os.path.exists(src_ckpt):
                shutil.copy2(src_ckpt, os.path.join(run_dir, fname))
        continue

    print("\n" + "=" * 80)
    print(f"🚀 開始訓練 {exp['label']} | img={exp['img_size']} | batch={exp['batch']} | epochs={exp['epochs']}")
    print("=" * 80)

    !rm -rf {run_dir}*
    !python train.py       --workers 4       --device 0       --batch-size {exp['batch']}       --epochs {exp['epochs']}       --data {exp['data']}       --img {exp['img_size']} {exp['img_size']}       --cfg {exp['cfg']}       --weights {exp['weights']}       --name {exp['name']}       --hyp {exp['hyp']}       --exist-ok

    # 即時備份至 Google Drive 防止斷線遺失
    os.makedirs(ckpt_dst, exist_ok=True)
    for fname in ["weights/best.pt", "results.txt", "results.png", "confusion_matrix.png"]:
        src_file = os.path.join(run_dir, fname)
        if os.path.exists(src_file):
            shutil.copy2(src_file, os.path.join(ckpt_dst, os.path.basename(fname)))
    print(f"💾 已即時備份 {exp['id']} 權重與日誌至 {ckpt_dst}")

# **Step 6: Validation-First 最佳門檻搜尋與 Test 集獨立盲測評估**
1. 先於 **Val 驗證集 (91 張)** 上掃描信心門檻，選出各實驗最佳 F1-Score 的 `best_conf`（嚴格避免 Test 資料洩漏）。
2. 再將各實驗權重於 **Test 測試集 (47 張)** 執行 `test.py --task test`，記錄 `mAP@0.5`、`mAP@0.5:0.95`、`Precision`、`Recall`、`F1` 與推論速度（`ms/img` & `FPS`）。

In [ ]:
import os
import re
import glob
import json
import subprocess
import sys
import numpy as np

sys.path.insert(0, "/content/workspace/dataset_repo/scripts")
from analyze_test_results import analyze_predictions, load_classes

%cd /content/workspace/yolov7
CLASSES = load_classes("/content/workspace/dataset_repo/data/dataset/classes.txt")
VAL_GT_STD = "/content/workspace/dataset_repo/data/dataset/val/labels"
VAL_GT_CLAHE = "/content/workspace/dataset_repo/data/dataset_clahe/val/labels"
VAL_IMG_STD = "/content/workspace/dataset_repo/data/dataset/val/images"
VAL_IMG_CLAHE = "/content/workspace/dataset_repo/data/dataset_clahe/val/images"

EVAL_RESULTS = {}

def find_best_conf_on_val(weight_path, exp):
    val_img = VAL_IMG_CLAHE if exp["is_clahe"] else VAL_IMG_STD
    val_gt = VAL_GT_CLAHE if exp["is_clahe"] else VAL_GT_STD
    det_name = f"val_sweep_{exp['id']}"
    !rm -rf runs/detect/{det_name}*
    !python detect.py --weights {weight_path} --conf 0.05 --img-size {exp['img_size']} --source {val_img} --name {det_name} --save-txt --save-conf --nosave --exist-ok > /dev/null 2>&1

    pred_dir = f"runs/detect/{det_name}/labels"
    best_conf, best_f1 = 0.25, -1.0
    if not os.path.exists(pred_dir):
        return best_conf, 0.0

    # 以不同信心門檻過濾預測框並計算 Val F1
    import tempfile, shutil
    for conf_t in np.round(np.linspace(0.10, 0.70, 13), 2):
        with tempfile.TemporaryDirectory() as tmpdir:
            for fname in os.listdir(pred_dir):
                if not fname.endswith(".txt"):
                    continue
                kept = []
                with open(os.path.join(pred_dir, fname), "r") as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) >= 6 and float(parts[5]) >= conf_t:
                            kept.append(line)
                with open(os.path.join(tmpdir, fname), "w") as f:
                    f.writelines(kept)
            stats, _ = analyze_predictions(val_gt, tmpdir, val_img, CLASSES, iou_thresh=0.5)
            tp, fp, fn = stats["tp"], stats["fp"], stats["fn"]
            prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
            if f1 > best_f1:
                best_f1 = f1
                best_conf = float(conf_t)
    return best_conf, best_f1

for exp in EXPERIMENTS:
    if not exp["enabled"]:
        continue
    weight_candidates = sorted(glob.glob(f"runs/train/{exp['name']}*/weights/best.pt"), key=os.path.getmtime)
    drive_ckpt_pt = os.path.join(DRIVE_CKPT_ROOT, exp["id"], "best.pt")
    if weight_candidates:
        weight_path = weight_candidates[-1]
    elif os.path.exists(drive_ckpt_pt):
        weight_path = drive_ckpt_pt
    else:
        print(f"⚠️ 找不到 {exp['id']} 權重，略過評估。")
        continue

    best_conf, val_f1 = find_best_conf_on_val(weight_path, exp)
    print("=" * 80)
    print(f"🎯 {exp['label']} | Val 最佳門檻 best_conf = {best_conf:.2f} (Val F1={val_f1:.3f})")

    test_name = f"{exp['id']}_test_evaluation"
    !rm -rf runs/test/{test_name}*
    cmd = [
        "python", "test.py",
        "--data", exp["data"],
        "--img", str(exp["img_size"]),
        "--batch", "8",
        "--conf", "0.001",
        "--iou", "0.65",
        "--device", "0",
        "--weights", weight_path,
        "--name", test_name,
        "--task", "test",
        "--verbose",
        "--save-txt",
        "--save-conf",
        "--exist-ok",
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)

    # 解析 all 類別指標與推論速度
    p_val, r_val, m50_val, m95_val, ms_val, fps_val, f1_val = "-", "-", "-", "-", "-", "-", "-"
    for line in proc.stdout.splitlines():
        m_metric = re.search(r"^\s*all\s+\d+\s+\d+\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)", line)
        if m_metric:
            p, r, m50, m95 = map(float, m_metric.groups())
            f1 = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0
            p_val, r_val, m50_val, m95_val, f1_val = f"{p:.3f}", f"{r:.3f}", f"{m50:.3f}", f"{m95:.3f}", f"{f1:.3f}"
        m_speed = re.search(r"Speed:\s*([0-9.]+)/([0-9.]+)/([0-9.]+)\s*ms", line)
        if m_speed:
            inf_ms = float(m_speed.group(1))
            total_ms = float(m_speed.group(3))
            ms_val = f"{total_ms:.1f}"
            fps_val = f"{1000.0 / total_ms:.1f}" if total_ms > 0 else "-"

    EVAL_RESULTS[exp["id"]] = {
        "weight_path": weight_path,
        "best_conf": best_conf,
        "precision": p_val,
        "recall": r_val,
        "map50": m50_val,
        "map50_95": m95_val,
        "f1": f1_val,
        "ms_per_img": ms_val,
        "fps": fps_val,
    }

# **Step 7: 套用 Val 最佳門檻於 Test 集執行推論 (`detect.py`)**
使用 Step 6 在 Val 集選出的 `best_conf` 對 47 張 Test 影像進行推論，供 Step 8 進行 4 維度誤差統計與並排視覺化對比。

In [ ]:
%cd /content/workspace/yolov7

TEST_IMG_STD = "/content/workspace/dataset_repo/data/dataset/test/images/"
TEST_IMG_CLAHE = "/content/workspace/dataset_repo/data/dataset_clahe/test/images/"

for exp in EXPERIMENTS:
    if not exp["enabled"] or exp["id"] not in EVAL_RESULTS:
        continue
    info = EVAL_RESULTS[exp["id"]]
    det_name = f"detect_test_{exp['id']}"
    src_dir = TEST_IMG_CLAHE if exp["is_clahe"] else TEST_IMG_STD
    !rm -rf runs/detect/{det_name}*
    print(f"🔎 推論 {exp['label']} | best_conf={info['best_conf']:.2f}")
    !python detect.py       --weights {info['weight_path']}       --conf {info['best_conf']}       --img-size {exp['img_size']}       --source {src_dir}       --name {det_name}       --save-txt       --save-conf       --exist-ok

# **Step 8: 成果總對照表、4 象限收斂圖、4 維度誤差統計與推論對比矩陣**
呼叫升級後的 `scripts/analyze_test_results.py --manifest` 自動產出：
1. `master_benchmark.csv`：含 `mAP@0.5`、`mAP@0.5:0.95`、`Precision`、`Recall`、`F1`、`ms/img`、`FPS` 與 `TP / FP / FN / Misclassification / Low IoU`。
2. `convergence_quadrants.png`：4 象限收斂曲線對照圖。
3. `inference_comparison_grid.jpg`：難例（含 `Sign_limit`、`two_stage_left_turn_box`、`crosswalk`）同圖並排推論對比矩陣。

In [ ]:
import json
import os
import matplotlib.pyplot as plt
import cv2

%cd /content/workspace/dataset_repo

manifest_exps = []
for exp in EXPERIMENTS:
    if not exp["enabled"]:
        continue
    ev = EVAL_RESULTS.get(exp["id"], {})
    results_txt = f"/content/workspace/yolov7/runs/train/{exp['name']}/results.txt"
    detect_labels = f"/content/workspace/yolov7/runs/detect/detect_test_{exp['id']}/labels"
    manifest_exps.append({
        "id": exp["id"],
        "label": exp["label"],
        "short_label": exp["short_label"],
        "arch": exp["arch"],
        "img_size": exp["img_size"],
        "batch": exp["batch"],
        "epochs": exp["epochs"],
        "results_txt": results_txt,
        "detect_labels_dir": detect_labels,
        "gt_dir": "data/dataset_clahe/test/labels" if exp["is_clahe"] else "data/dataset/test/labels",
        "images_dir": "data/dataset_clahe/test/images" if exp["is_clahe"] else "data/dataset/test/images",
        "best_conf": ev.get("best_conf", "-"),
        "map50": ev.get("map50", "-"),
        "map50_95": ev.get("map50_95", "-"),
        "precision": ev.get("precision", "-"),
        "recall": ev.get("recall", "-"),
        "f1": ev.get("f1", "-"),
        "ms_per_img": ev.get("ms_per_img", "-"),
        "fps": ev.get("fps", "-"),
    })

os.makedirs("results/benchmark", exist_ok=True)
manifest_path = "results/benchmark/benchmark_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump({
        "gt_dir": "data/dataset/test/labels",
        "images_dir": "data/dataset/test/images",
        "experiments": manifest_exps,
    }, f, ensure_ascii=False, indent=2)

!python scripts/analyze_test_results.py   --manifest {manifest_path}   --output-dir results/benchmark   --classes data/dataset/classes.txt   --iou 0.5   --max-grid-images 5

for img_name, title in [
    ("results/benchmark/convergence_quadrants.png", "4-Quadrant Convergence Curves"),
    ("results/benchmark/inference_comparison_grid.jpg", "Side-by-Side Inference Comparison Grid"),
]:
    if os.path.exists(img_name):
        img = cv2.imread(img_name)
        plt.figure(figsize=(18, 11), dpi=130)
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(title, fontsize=14)
        plt.axis("off")
        plt.tight_layout()
        plt.show()

# **Step 9: 儲存完整權重、圖表與基準報告至 Google Drive**

In [ ]:
%cd /content/workspace/yolov7
DRIVE_OUT = "/content/drive/MyDrive/NCHU_YOLO_Project_Traffic_Facility_6_classes"
!mkdir -p {DRIVE_OUT}/results/benchmark
!cp -r /content/workspace/dataset_repo/results/benchmark/* {DRIVE_OUT}/results/benchmark/

!rm -f {DRIVE_OUT}/results_summary_458_exp1_to_10.zip
!zip -r {DRIVE_OUT}/results_summary_458_exp1_to_10.zip   runs/train/exp*   runs/test/exp*   runs/detect/detect_test_*   /content/workspace/dataset_repo/results/benchmark/

print("🎉 完整 Exp 1~10 實驗權重、Benchmark CSV、4象限圖與推論對比矩陣已成功備份至 Google Drive！")

# **Step 10: 動態推論對比影片製作 (`Exp 7` vs `Exp 8` Side-by-Side Demo for PPT)**
> 💡 **免重訓快速通道**：由於 Google Drive 的 `checkpoints/` 已保留 Exp 1~10 權重，若僅需產出簡報用動態推論對比影片，**只要執行 Step 1 → Step 2 → Step 3，即可直接跳至此步驟執行**（無須執行 Step 4~9）。

本步驟會自動完成：
1. 直接載入 Google Drive 的 `checkpoints/exp7/best.pt`（640 解析度，速度優先）與 `checkpoints/exp8/best.pt`（960 解析度，精度優先）。
2. 使用 `detect.py` 對測試行車記錄器影片分別執行推論，輸出 `demo_exp7` 與 `demo_exp8` 標註影片。
3. 使用 `ffmpeg` 水平拼接為左右並排對比影片 `comparison_demo.mp4`（採用 `H.264 / yuv420p` 編碼，確保嵌入 PowerPoint 後 100% 穩定播放）以及可選的動態 GIF（`comparison_demo.gif`）。
4. **另行獨立存放**：將所有動態推論結果存至 Google Drive 專屬資料夾 `NCHU_YOLO_Project_Traffic_Facility_6_classes/demo_videos/`，不覆蓋原有 `checkpoints/` 與 `results/`。

In [ ]:
import os
import glob
import shutil
import subprocess

os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
%cd /content/workspace/yolov7

# ===== 路徑與參數設定 =====
DRIVE_ROOT = "/content/drive/MyDrive/NCHU_YOLO_Project_Traffic_Facility_6_classes"
DRIVE_CKPT_ROOT = os.path.join(DRIVE_ROOT, "checkpoints")
DRIVE_DEMO_OUT = os.path.join(DRIVE_ROOT, "demo_videos")  # 動態推論結果另行獨立存放目錄
os.makedirs(DRIVE_DEMO_OUT, exist_ok=True)

# 測試影片路徑：設為 None 時會依序自動搜尋 Google Drive sample_videos -> repo sample_videos -> 跳出 Colab 上傳視窗
CUSTOM_VIDEO_PATH = None  # 例如: f"{DRIVE_ROOT}/sample_videos/dashcam.mp4"

CONF_THRES_EXP7 = 0.25
CONF_THRES_EXP8 = 0.25
ADD_LABEL_BANNER = True   # 在左右畫面左上角加上 Exp 7 (640px) / Exp 8 (960px) 標題列
EXPORT_GIF = True         # 是否額外轉出輕量動態 GIF (comparison_demo.gif)
CLIP_START_SEC = None     # 若只想取片段可設定起始秒數（如 0），None 表示不裁切
CLIP_DURATION_SEC = None  # 若只想取片段可設定長度秒數（如 15），None 表示完整影片

# 1. 優先讀取 Google Drive checkpoints 權重（跳過訓練直接推論），找不到才回退至本地 runs/train
def resolve_weight(exp_id, local_run_name):
    drive_pt = os.path.join(DRIVE_CKPT_ROOT, exp_id, "best.pt")
    if os.path.exists(drive_pt):
        return drive_pt
    local_candidates = sorted(glob.glob(f"runs/train/{local_run_name}*/weights/best.pt"), key=os.path.getmtime)
    if local_candidates:
        return local_candidates[-1]
    raise FileNotFoundError(f"❌ 找不到 {exp_id} 權重！請確認 {drive_pt} 是否存在。")

EXP7_WEIGHTS = resolve_weight("exp7", "exp7_yolov7_base_100ep_default")
EXP8_WEIGHTS = resolve_weight("exp8", "exp8_yolov7_base_960_highres")
print(f"✅ Exp 7 權重: {EXP7_WEIGHTS}")
print(f"✅ Exp 8 權重: {EXP8_WEIGHTS}")

# 2. 取得測試影片來源
def find_video_in_dir(directory):
    if not os.path.isdir(directory):
        return None
    for ext in ("*.mp4", "*.MP4", "*.mov", "*.MOV", "*.avi", "*.AVI", "*.mkv"):
        matches = sorted(glob.glob(os.path.join(directory, ext)))
        if matches:
            return matches[0]
    return None

source_video = None
if CUSTOM_VIDEO_PATH and os.path.exists(CUSTOM_VIDEO_PATH):
    source_video = CUSTOM_VIDEO_PATH
else:
    for candidate_dir in [
        os.path.join(DRIVE_ROOT, "sample_videos"),
        "/content/workspace/dataset_repo/sample_videos",
        "/content/workspace/yolov7/sample_videos",
    ]:
        found = find_video_in_dir(candidate_dir)
        if found:
            source_video = found
            break

if source_video is None:
    print("📤 未在預設目錄找到測試影片，請直接上傳一段行車記錄器測試短片 (.mp4 / .mov)：")
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("❌ 未上傳任何影片，中止動態推論。")
    upload_dir = "/content/workspace/demo_input"
    os.makedirs(upload_dir, exist_ok=True)
    uploaded_name = next(iter(uploaded.keys()))
    source_video = os.path.join(upload_dir, uploaded_name)
    shutil.move(uploaded_name, source_video)

print(f"🎬 來源影片: {source_video}")

# 可選：若設定了裁切區間，先擷取短片以加速推論並縮小 PPT 檔案體積
video_to_infer = source_video
video_stem = os.path.splitext(os.path.basename(source_video))[0]
if CLIP_START_SEC is not None or CLIP_DURATION_SEC is not None:
    clipped_path = f"/content/workspace/{video_stem}_clip.mp4"
    trim_cmd = ["ffmpeg", "-y"]
    if CLIP_START_SEC is not None:
        trim_cmd += ["-ss", str(CLIP_START_SEC)]
    trim_cmd += ["-i", source_video]
    if CLIP_DURATION_SEC is not None:
        trim_cmd += ["-t", str(CLIP_DURATION_SEC)]
    trim_cmd += ["-c:v", "libx264", "-crf", "20", "-preset", "veryfast", "-an", clipped_path]
    subprocess.run(trim_cmd, check=True)
    video_to_infer = clipped_path
    video_stem = f"{video_stem}_clip"
    print(f"✂️ 已裁切測試短片: {video_to_infer}")

# 3. 分別執行 Exp 7 (640px) 與 Exp 8 (960px) 影片推論
!rm -rf runs/detect/demo_exp7 runs/detect/demo_exp8

print("\n" + "=" * 80)
print(f"🚀 [1/2] 執行 Exp 7 推論 (640 解析度，速度優先 | conf={CONF_THRES_EXP7})")
print("=" * 80)
!python detect.py --weights "{EXP7_WEIGHTS}" --source "{video_to_infer}" --img-size 640 --conf-thres {CONF_THRES_EXP7} --name demo_exp7 --exist-ok

print("\n" + "=" * 80)
print(f"🚀 [2/2] 執行 Exp 8 推論 (960 解析度，精度優先 | conf={CONF_THRES_EXP8})")
print("=" * 80)
!python detect.py --weights "{EXP8_WEIGHTS}" --source "{video_to_infer}" --img-size 960 --conf-thres {CONF_THRES_EXP8} --name demo_exp8 --exist-ok

def find_detected_video(detect_dir):
    found = find_video_in_dir(detect_dir)
    if not found:
        raise FileNotFoundError(f"❌ 在 {detect_dir} 找不到推論輸出影片！")
    return found

exp7_raw_vid = find_detected_video("runs/detect/demo_exp7")
exp8_raw_vid = find_detected_video("runs/detect/demo_exp8")

# 4. 使用 ffmpeg 製作左右並排 (Side-by-Side) 對比影片 (H.264 + yuv420p 確保 PPT 100% 相容)
local_demo_dir = "/content/workspace/demo_outputs"
os.makedirs(local_demo_dir, exist_ok=True)
comparison_mp4 = os.path.join(local_demo_dir, f"comparison_demo_{video_stem}.mp4")

font_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
if ADD_LABEL_BANNER and os.path.exists(font_path):
    filter_complex = (
        f"[0:v]drawtext=fontfile={font_path}:text='Exp 7\\: Base-640 (Speed)':"
        f"x=24:y=24:fontsize=34:fontcolor=white:box=1:boxcolor=black@0.65:boxborderw=10[left];"
        f"[1:v]drawtext=fontfile={font_path}:text='Exp 8\\: Base-960 (Accuracy)':"
        f"x=24:y=24:fontsize=34:fontcolor=yellow:box=1:boxcolor=black@0.65:boxborderw=10[right];"
        f"[left]pad=iw*2:ih[int];[int][right]overlay=W/2:0"
    )
else:
    filter_complex = "[0:v]pad=iw*2:ih[int];[int][1:v]overlay=W/2:0"

print("\n🎬 使用 ffmpeg 合成左右並排對比影片...")
ffmpeg_sbs_cmd = [
    "ffmpeg", "-y",
    "-i", exp7_raw_vid,
    "-i", exp8_raw_vid,
    "-filter_complex", filter_complex,
    "-c:v", "libx264", "-crf", "23", "-preset", "veryfast",
    "-pix_fmt", "yuv420p", "-movflags", "+faststart",
    comparison_mp4,
]
subprocess.run(ffmpeg_sbs_cmd, check=True)

# 將單獨的 Exp 7 / Exp 8 推論影片也轉成 PPT 相容的 H.264 格式一併備份
exp7_h264 = os.path.join(local_demo_dir, f"demo_exp7_{video_stem}.mp4")
exp8_h264 = os.path.join(local_demo_dir, f"demo_exp8_{video_stem}.mp4")
for src_v, dst_v in [(exp7_raw_vid, exp7_h264), (exp8_raw_vid, exp8_h264)]:
    subprocess.run([
        "ffmpeg", "-y", "-i", src_v,
        "-c:v", "libx264", "-crf", "23", "-preset", "veryfast",
        "-pix_fmt", "yuv420p", "-movflags", "+faststart", dst_v
    ], check=True)

# 5. 可選：同步轉出輕量動態 GIF
saved_files = [exp7_h264, exp8_h264, comparison_mp4]
if EXPORT_GIF:
    comparison_gif = os.path.join(local_demo_dir, f"comparison_demo_{video_stem}.gif")
    print("🎞️ 同步產生輕量動態 GIF (寬度 960px, 10 FPS)...")
    gif_filter = "fps=10,scale=960:-1:flags=lanczos,split[s0][s1];[s0]palettegen[p];[s1][p]paletteuse"
    subprocess.run([
        "ffmpeg", "-y", "-i", comparison_mp4,
        "-vf", gif_filter, "-loop", "0", comparison_gif
    ], check=True)
    saved_files.append(comparison_gif)

# 6. 另行存放至 Google Drive 獨立目錄 demo_videos/
print("\n" + "=" * 80)
print(f"💾 另行存放動態推論結果至 Google Drive: {DRIVE_DEMO_OUT}")
print("=" * 80)
for local_f in saved_files:
    dst_f = os.path.join(DRIVE_DEMO_OUT, os.path.basename(local_f))
    shutil.copy2(local_f, dst_f)
    size_mb = os.path.getsize(dst_f) / (1024 * 1024)
    print(f"  ✅ {os.path.basename(dst_f)} ({size_mb:.2f} MB) -> {dst_f}")

# 同時保留固定檔名 comparison_demo.mp4 方便直接取用
latest_sbs_drive = os.path.join(DRIVE_DEMO_OUT, "comparison_demo.mp4")
shutil.copy2(comparison_mp4, latest_sbs_drive)
print(f"  ⭐ 最新並排對比影片快速路徑: {latest_sbs_drive}")

In [ ]:
# （可選）直接從 Colab 下載 comparison_demo.mp4 至本機，方便立即嵌入 PowerPoint
from google.colab import files
import os

latest_sbs_drive = "/content/drive/MyDrive/NCHU_YOLO_Project_Traffic_Facility_6_classes/demo_videos/comparison_demo.mp4"
if os.path.exists(latest_sbs_drive):
    files.download(latest_sbs_drive)
else:
    print("⚠️ 尚未產生 comparison_demo.mp4，請先執行上一個 Step 10 儲存格。")